## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:郑灏昕


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

## B 长跑

In [ ]:
import sys
import heapq


def can_run(n, L, maxn, S, stations):
    best = {}
    for p, c in stations:
        if p not in best or c < best[p]:
            best[p] = c

    points = [(0, 0)]
    for p in sorted(best):
        if 0 < p < L:
            points.append((p, best[p]))
    points.append((L, 0))

    inf = 10 ** 18
    dp = [inf] * len(points)
    dp[0] = 0
    heap = [(0, 0)]

    for i in range(1, len(points)):
        while heap and points[i][0] - points[heap[0][1]][0] > maxn:
            heapq.heappop(heap)

        if heap:
            dp[i] = heap[0][0]

        if i != len(points) - 1 and dp[i] <= S:
            heapq.heappush(heap, (dp[i] + points[i][1], i))

    return dp[-1] <= S


def solve():
    data = list(map(int, sys.stdin.buffer.read().split()))
    ans = []
    i = 0

    while i < len(data):
        n, L, maxn, S = data[i:i + 4]
        i += 4
        stations = []
        for _ in range(n):
            stations.append((data[i], data[i + 1]))
            i += 2
        ans.append("Yes" if can_run(n, L, maxn, S, stations) else "No")

    sys.stdout.write("\n".join(ans))


solve()

## C 最长回文

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

const int MOD1 = 1e9 + 7;
const int MOD2 = 1e9 + 9;
const int B1 = 313;
const int B2 = 317;

struct Hash {
    long long h1, h2;
    bool operator==(const Hash& o) const {
        return h1 == o.h1 && h2 == o.h2;
    }
};

int n;
string A, B, A_R;

long long p1_pow[100005], p2_pow[100005];
long long h1_A[100005], h2_A[100005];
long long h1_B[100005], h2_B[100005];

// 预处理字符串 Hash 前缀数组
void init_hash() {
    p1_pow[0] = 1; p2_pow[0] = 1;
    for (int i = 1; i <= n; i++) {
        p1_pow[i] = (p1_pow[i - 1] * B1) % MOD1;
        p2_pow[i] = (p2_pow[i - 1] * B2) % MOD2;
    }
    for (int i = 1; i <= n; i++) {
        h1_A[i] = (h1_A[i - 1] * B1 + A_R[i - 1]) % MOD1;
        h2_A[i] = (h2_A[i - 1] * B2 + A_R[i - 1]) % MOD2;
    }
    for (int i = 1; i <= n; i++) {
        h1_B[i] = (h1_B[i - 1] * B1 + B[i - 1]) % MOD1;
        h2_B[i] = (h2_B[i - 1] * B2 + B[i - 1]) % MOD2;
    }
}

// 提取 A 逆序串的区间 Hash
Hash get_hash_A(int l, int r) {
    int len = r - l + 1;
    long long res1 = (h1_A[r] - h1_A[l - 1] * p1_pow[len]) % MOD1;
    if (res1 < 0) res1 += MOD1;
    long long res2 = (h2_A[r] - h2_A[l - 1] * p2_pow[len]) % MOD2;
    if (res2 < 0) res2 += MOD2;
    return {res1, res2};
}

// 提取 B 原串的区间 Hash
Hash get_hash_B(int l, int r) {
    int len = r - l + 1;
    long long res1 = (h1_B[r] - h1_B[l - 1] * p1_pow[len]) % MOD1;
    if (res1 < 0) res1 += MOD1;
    long long res2 = (h2_B[r] - h2_B[l - 1] * p2_pow[len]) % MOD2;
    if (res2 < 0) res2 += MOD2;
    return {res1, res2};
}

// 二分查找 LCP (最长公共前缀)
int get_lcp(int i, int j) {
    int low = 1, high = min(n - i + 1, n - j + 1);
    int ans = 0;
    while (low <= high) {
        int mid = low + (high - low) / 2;
        if (get_hash_A(i, i + mid - 1) == get_hash_B(j, j + mid - 1)) {
            ans = mid;
            low = mid + 1;
        } else {
            high = mid - 1;
        }
    }
    return ans;
}

// Manacher 算法求解回文半径
vector<int> manacher(string s) {
    int len = s.length();
    string t = "#";
    for (int i = 0; i < len; i++) {
        t += s[i];
        t += "#";
    }
    int m = t.length();
    vector<int> rad(m, 0);
    int c = 0, r = 0;
    for (int i = 0; i < m; i++) {
        int mirror = 2 * c - i;
        if (i < r) rad[i] = min(r - i, rad[mirror]);
        while (i - rad[i] - 1 >= 0 && i + rad[i] + 1 < m && t[i - rad[i] - 1] == t[i + rad[i] + 1]) {
            rad[i]++;
        }
        if (i + rad[i] > r) {
            c = i;
            r = i + rad[i];
        }
    }
    return rad;
}

int main() {
    // 基础 I/O 提速
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    if (!(cin >> n)) return 0;
    cin >> A >> B;

    A_R = A;
    reverse(A_R.begin(), A_R.end());

    init_hash();

    // 预处理两者的 Manacher 半径，全部映射为 1-based (长度为 2n+1)
    vector<int> rad0_A = manacher(A);
    vector<int> rad_A(2 * n + 2, 0);
    for (int i = 0; i <= 2 * n; i++) rad_A[i + 1] = rad0_A[i];

    vector<int> rad0_B = manacher(B);
    vector<int> rad_B(2 * n + 2, 0);
    for (int i = 0; i <= 2 * n; i++) rad_B[i + 1] = rad0_B[i];

    int ans = 0;
    // 兜底情况：直接在 A 或 B 单独提取出的最长回文子串
    for (int i = 1; i <= 2 * n + 1; i++) {
        ans = max(ans, rad_A[i]);
        ans = max(ans, rad_B[i]);
    }

    // Case 1: 对称中心存在于 A 的部分中
    for (int c = 1; c <= 2 * n + 1; c++) {
        int i = min(n, (c + rad_A[c]) / 2);  // 贪心：将断点拉至极右极限
        if (i < 1 || i < c / 2) continue;
        int idx_A = n - c + i + 2;
        int idx_B = i;
        int l = 0;
        if (idx_A >= 1 && idx_A <= n && idx_B >= 1 && idx_B <= n) {
            l = get_lcp(idx_A, idx_B);
        }
        ans = max(ans, 2 * i - c + 1 + 2 * l);
    }

    // Case 2: 对称中心存在于 B 的部分中
    for (int c = 1; c <= 2 * n + 1; c++) {
        int i = max(1, (c - rad_B[c] + 1) / 2); // 贪心：将断点拉至极左极限
        if (i > n || i > (c + 1) / 2) continue;
        int idx_A = n - i + 1;
        int idx_B = c - i + 1;
        int l = 0;
        if (idx_A >= 1 && idx_A <= n && idx_B >= 1 && idx_B <= n) {
            l = get_lcp(idx_A, idx_B);
        }
        ans = max(ans, c - 2 * i + 1 + 2 * l);
    }

    cout << ans << "\n";
    return 0;
}

## D 优惠券

In [ ]:
import sys


def check(m, tokens, pos):
    state = {}
    unknown = 0

    for line in range(1, m + 1):
        op = tokens[pos]
        pos += 1

        if op == "?" or op == "？":
            unknown += 1
            continue

        x = int(tokens[pos])
        pos += 1
        old = state.get(x, 0)

        if op == "I":
            if old != 0:
                return line, pos
            state[x] = 1
        else:
            if old == 1:
                state[x] = 2
            elif old == 0 and unknown > 0:
                unknown -= 1
                state[x] = 2
            else:
                return line, pos

    return -1, pos


def solve():
    tokens = sys.stdin.read().split()
    ans = []
    pos = 0

    while pos < len(tokens):
        m = int(tokens[pos])
        pos += 1
        bad, pos = check(m, tokens, pos)
        ans.append(str(bad))

    print("\n".join(ans))


solve()

## E 任意点

In [ ]:
import sys


class DSU:
    def __init__(self, n):
        self.fa = list(range(n))

    def find(self, x):
        while self.fa[x] != x:
            self.fa[x] = self.fa[self.fa[x]]
            x = self.fa[x]
        return x

    def union(self, x, y):
        fx = self.find(x)
        fy = self.find(y)
        if fx != fy:
            self.fa[fx] = fy


def solve():
    data = list(map(int, sys.stdin.read().split()))
    if not data:
        return

    n = data[0]
    point = []
    idx = 1
    for _ in range(n):
        x = data[idx]
        y = data[idx + 1]
        idx += 2
        point.append((x, y))

    dsu = DSU(n)
    last_x = {}
    last_y = {}

    for i, (x, y) in enumerate(point):
        if x in last_x:
            dsu.union(i, last_x[x])
        else:
            last_x[x] = i

        if y in last_y:
            dsu.union(i, last_y[y])
        else:
            last_y[y] = i

    cnt = len({dsu.find(i) for i in range(n)})
    print(cnt - 1)


solve()

## F 通配符匹配

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <cstring>

using namespace std;

// 定义模式串被拆分后的单元块
struct Token {
    int type; // 0: 普通纯字母字符串, 1: 星号(*), 2: 问号(?)
    string str;
    vector<int> lps; // 用于 KMP 算法的 lps 数组
};

// 预处理 KMP 算法的 LPS (最长公共前后缀) 数组，保证严格 O(M)
vector<int> build_lps(const string& P) {
    int m = P.length();
    vector<int> lps(m, 0);
    int len = 0;
    int i = 1;
    while (i < m) {
        if (P[i] == P[len]) {
            len++;
            lps[i] = len;
            i++;
        } else {
            if (len != 0) {
                len = lps[len - 1];
            } else {
                lps[i] = 0;
                i++;
            }
        }
    }
    return lps;
}

const int MAXN = 100005;
// 使用 char 数组替代 vector<bool> / bitset，避免位运算带来的额外开销，直接操控字节达到最高速
char dp[MAXN];
char dp_next[MAXN];
char match_pos[MAXN];

int main() {
    // 终极 I/O 加速
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    string pattern;
    if (!(cin >> pattern)) return 0;

    int n;
    if (!(cin >> n)) return 0;

    vector<Token> tokens;
    int n_pat = pattern.length();
    int i = 0;

    // 1. 将模式串解析为一个个 Token
    while (i < n_pat) {
        if (pattern[i] == '*') {
            // 连续的 * 等同于一个 *，直接贪心合并
            while (i < n_pat && pattern[i] == '*') i++;
            tokens.push_back({1, "", {}});
        } else if (pattern[i] == '?') {
            tokens.push_back({2, "", {}});
            i++;
        } else {
            string s = "";
            // 提取出纯字母段
            while (i < n_pat && pattern[i] != '*' && pattern[i] != '?') {
                s += pattern[i];
                i++;
            }
            // 提前为纯字母段计算好 KMP 的 LPS，后续文件不管有多少个，直接复用！
            tokens.push_back({0, s, build_lps(s)});
        }
    }

    // 2. 流水线式处理每一个文件
    for (int k = 0; k < n; ++k) {
        string S;
        cin >> S;
        int N = S.length();
        if (N >= MAXN) N = MAXN - 1; // 边界保护

        // 初始化：什么都没匹配时，只有起点是点亮的
        memset(dp, 0, N + 1);
        dp[0] = 1;

        // 依次用模式串的每一个片段推进状态
        for (const auto& token : tokens) {
            memset(dp_next, 0, N + 1);

            if (token.type == 1) {
                // 处理 '*'：只要前面某个点能抵达，* 可以延伸到后面所有的位置
                int first_p = -1;
                for (int p = 0; p <= N; ++p) {
                    if (dp[p]) {
                        first_p = p;
                        break;
                    }
                }
                if (first_p != -1) {
                    // 使用底层的 memset 极速将后面的全部合法点铺满
                    memset(dp_next + first_p, 1, N - first_p + 1);
                }
            }
            else if (token.type == 2) {
                // 处理 '?'：精确吃掉一个字符，所有状态整体右移 1 格
                for (int p = 0; p < N; ++p) {
                    if (dp[p]) {
                        dp_next[p + 1] = 1;
                    }
                }
            }
            else {
                // 处理纯字母字符串：利用 KMP 找出这个串在 S 中所有的出现位置，绝对保证 O(N) 且不回溯
                memset(match_pos, 0, N + 1);
                int L = token.str.length();
                int idx = 0, j = 0;

                while (idx < N) {
                    if (token.str[j] == S[idx]) {
                        j++;
                        idx++;
                    }
                    if (j == L) {
                        match_pos[idx - j] = 1; // 记录匹配成功的起始点
                        j = token.lps[j - 1];
                    } else if (idx < N && token.str[j] != S[idx]) {
                        if (j != 0) j = token.lps[j - 1];
                        else idx++;
                    }
                }

                // 将所有符合条件的路径向右推进 L 格
                for (int p = 0; p <= N - L; ++p) {
                    if (dp[p] && match_pos[p]) {
                        dp_next[p + L] = 1;
                    }
                }
            }

            // 状态滚动
            memcpy(dp, dp_next, N + 1);
        }

        // 如果最终状态能够正好推进到文件末尾 N，说明完美匹配！
        if (dp[N]) cout << "YES\n";
        else cout << "NO\n";
    }

    return 0;
}

## G 汉诺塔

In [ ]:
import sys


def simulate(n, order):
    mp = {"A": 0, "B": 1, "C": 2}
    peg = [list(range(n, 0, -1)), [], []]
    last = 0
    ans = 0

    while True:
        if len(peg[1]) == n or len(peg[2]) == n:
            return ans

        for op in order:
            u = mp[op[0]]
            v = mp[op[1]]

            if not peg[u]:
                continue
            if peg[u][-1] == last:
                continue
            if peg[v] and peg[v][-1] < peg[u][-1]:
                continue

            last = peg[u][-1]
            peg[v].append(peg[u].pop())
            ans += 1
            break


def solve():
    data = sys.stdin.read().split()
    n = int(data[0])
    order = data[1:]

    if n <= 7:
        print(simulate(n, order))
        return

    flag = simulate(7, order)
    if flag == 127:
        print((1 << n) - 1)
    elif flag == 729:
        print(3 ** (n - 1))
    else:
        print(2 * (3 ** (n - 1)) - 1)


solve()

## H 马步距离

In [ ]:
import sys


def dis(x1, y1, x2, y2):
    x = abs(x1 - x2)
    y = abs(y1 - y2)

    if x < y:
        x, y = y, x

    if x == 1 and y == 0:
        return 3
    if x == 2 and y == 2:
        return 4

    ans = max((x + 1) // 2, (x + y + 2) // 3)
    if (ans + x + y) % 2 == 1:
        ans += 1

    return ans


data = list(map(int, sys.stdin.read().split()))
print(dis(data[0], data[1], data[2], data[3]))

## I 直方图最大矩形

In [ ]:
from typing import List


class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        stack = []
        ans = 0
        arr = heights + [0]

        for i, h in enumerate(arr):
            while stack and arr[stack[-1]] > h:
                height = arr[stack.pop()]
                if stack:
                    width = i - stack[-1] - 1
                else:
                    width = i
                area = height * width
                if area > ans:
                    ans = area
            stack.append(i)

        return ans

## J 消防局的设立

In [ ]:
import sys
from collections import deque


def solve():
    data = list(map(int, sys.stdin.read().split()))
    if not data:
        return

    n = data[0]
    parent = [0] * (n + 1)
    children = [[] for _ in range(n + 1)]
    graph = [[] for _ in range(n + 1)]

    for i in range(2, n + 1):
        parent[i] = data[i - 1]
        children[parent[i]].append(i)
        graph[i].append(parent[i])
        graph[parent[i]].append(i)

    order = [1]
    for x in order:
        order.extend(children[x])

    covered = [False] * (n + 1)
    ans = 0

    def mark(start):
        q = deque([(start, 0)])
        seen = {start}
        while q:
            x, d = q.popleft()
            covered[x] = True
            if d == 2:
                continue
            for y in graph[x]:
                if y not in seen:
                    seen.add(y)
                    q.append((y, d + 1))

    for x in reversed(order):
        if covered[x]:
            continue

        station = x
        for _ in range(2):
            if parent[station]:
                station = parent[station]

        ans += 1
        mark(station)

    print(ans)


solve()